# ------------------ Data prepration for ML - classification model ------------------

####

# Project plan

#### Importing important packages

In [58]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import math as math
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline


from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder

from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler

######

#### Importing cleaned dataset which earlier saved in pickle file

In [59]:
df = pd.read_pickle(r"C:\Users\Naveen\Desktop\GitHub\TransGlobe_Business\Classification_model\df.pkl")

df.head()

,carrierid,deliverymode,shippingpriority,paymentmethod,weathercondition,trafficcondition,orderquantity,packageweightkg,packagevolumem3,distancekm,...,fueltype,vehicleageyears,carriertype,servicelevel,historicontimerate,customerlocation_zone,originlocation_zone,destinationlocation_zone,customertenureyears,late_delivery
0,9,Road,Express,COD,Clear,High,21,100.70,0.6140,826.9,...,CNG,8,Third-Party,Standard,0.75,Zone-A,Zone-A,Zone-C,3.846680,0
1,2,Sea,Express,Prepaid,Clear,Low,3,6.74,0.1811,1410.5,...,CNG,9,In-House,Express,0.78,Zone-A,Zone-A,Zone-C,1.911020,0
2,10,Road,Standard,COD,Clear,High,5,1.10,0.3300,1243.0,...,Petrol,1,Third-Party,Express,0.85,Zone-A,Zone-A,Zone-A,4.468172,0
3,5,Road,Same-Day,COD,Clear,High,23,35.91,1.8713,2066.3,...,Diesel,2,In-House,Standard,0.88,Zone-A,Zone-A,Zone-A,4.596851,1
4,8,Road,Standard,COD,Clear,High,21,229.65,2.4703,1930.8,...,Petrol,9,In-House,Premium,0.93,Zone-A,Zone-C,Zone-A,6.225873,1


In [60]:
# Checking datatypes once again for better undertanding

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20461 entries, 0 to 22109
Data columns (total 43 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   carrierid                  20461 non-null  object 
 1   deliverymode               20461 non-null  object 
 2   shippingpriority           20461 non-null  object 
 3   paymentmethod              20461 non-null  object 
 4   weathercondition           20151 non-null  object 
 5   trafficcondition           20461 non-null  object 
 6   orderquantity              20461 non-null  int64  
 7   packageweightkg            20054 non-null  float64
 8   packagevolumem3            19843 non-null  float64
 9   distancekm                 20461 non-null  float64
 10  shippingcost               20152 non-null  float64
 11  ordervalue                 20461 non-null  float64
 12  fuelconsumptionliters      19946 non-null  float64
 13  promiseddeliverydays       20461 non-null  int64  


In [61]:
# checking num of unique values for each column which have cardinality above 20 

df[df.columns[df.nunique() > 20]].nunique()

orderquantity               24
packageweightkg           9881
packagevolumem3          13980
distancekm               12855
shippingcost             19441
ordervalue               20108
fuelconsumptionliters    15031
warehousename               30
capacityunits               30
experienceyears            125
capacitykg                 497
customertenureyears       2447
dtype: int64

#####

# Define X and y

In [62]:
y_variable = "late_delivery"

x_variables = list(df.columns.difference(["late_delivery"]))

#####

# Train-test split

In [63]:
train_x, test_x, train_y, test_y = train_test_split(df.loc[:, x_variables], df.loc[:, y_variable], train_size=0.7, random_state=1)

In [64]:
print("Train: ")
display(train_x.head())

print("\n\nTest: ")
display(test_x.head())

print("\n\nTrain - y-variable: ")
display(train_y.head())

print("\n\nTest - y-variable: ")
display(test_y.head())

Train: 


,automationlevel,avgtrafficlevel,capacitykg,capacityunits,carrierid,carriertype,category,customerlocation_zone,customertenureyears,customertier,...,shifttype,shippingcost,shippingpriority,subcategory,trafficcondition,vehicleageyears,vehicletype,warehousename,warehousetype,weathercondition
12705,Low,High,1772.2,45212,10,Third-Party,Grocery,Zone-C,3.731691,Silver,...,Night,2872.15,Standard,Household Supplies,Medium,6,Two-Wheeler,Warehouse_015,Regional Hub,Storm
18766,High,Medium,1278.1,44670,10,Third-Party,Electronics,Zone-C,3.077344,Silver,...,Rotational,3061.70,Standard,Accessories,Low,11,Van,Warehouse_003,Regional Hub,Clear
10283,Medium,Low,1370.4,20672,3,Third-Party,Electronics,Zone-C,3.118412,Bronze,...,Night,2082.68,Express,Mobile Phones,Medium,0,Van,Warehouse_006,Regional Hub,Clear
3649,Low,High,860.5,12263,7,Third-Party,Grocery,Zone-A,2.171116,Silver,...,Day,2883.51,Standard,Household Supplies,High,5,Van,Warehouse_018,Fulfillment Center,Clear
5839,High,Low,1125.2,26718,5,In-House,Industrial,Zone-C,5.911020,Silver,...,Night,2351.55,Same-Day,Spare Parts,Low,7,Van,Warehouse_023,Regional Hub,Storm




Test: 


,automationlevel,avgtrafficlevel,capacitykg,capacityunits,carrierid,carriertype,category,customerlocation_zone,customertenureyears,customertier,...,shifttype,shippingcost,shippingpriority,subcategory,trafficcondition,vehicleageyears,vehicletype,warehousename,warehousetype,weathercondition
4054,Low,Medium,1649.8,39380,9,Third-Party,Industrial,Zone-A,5.431896,Silver,...,Day,945.84,Express,Machinery Components,High,0,Mini Truck,Warehouse_004,Regional Hub,Extreme Heat
4912,Low,High,558.9,27485,10,Third-Party,Electronics,Zone-C,3.063655,Bronze,...,Night,2507.95,Express,Laptops,Medium,5,Refrigerated Truck,Warehouse_011,Fulfillment Center,Rain
11193,Low,Medium,629.7,22991,2,In-House,Industrial,Zone-C,4.016427,Bronze,...,Night,2051.23,Express,Spare Parts,Low,0,Mini Truck,Warehouse_005,Regional Hub,Rain
16557,Low,High,1318.4,48336,8,In-House,Apparel,Zone-A,0.950034,Platinum,...,Day,3162.75,Standard,Womenswear,Medium,9,Mini Truck,Warehouse_022,Regional Hub,Clear
12632,High,High,1429.1,46733,1,Third-Party,Furniture,Zone-A,4.711841,Platinum,...,Night,3045.56,Same-Day,Home Furniture,Medium,8,Refrigerated Truck,Warehouse_017,Regional Hub,Clear




Train - y-variable: 


12705    1
18766    0
10283    0
3649     0
5839     1
Name: late_delivery, dtype: int64



Test - y-variable: 


4054     0
4912     0
11193    0
16557    0
12632    0
Name: late_delivery, dtype: int64

# Outlier treatment (Use IQR)

In [65]:
# Select numerical column names
numeric_columns = df.select_dtypes(include=["int", "float"]).columns.difference(["late_delivery"])
# Calculate IQR
Q1 = df[numeric_columns].quantile(0.25)
Q3 = df[numeric_columns].quantile(0.75)
IQR = Q3 - Q1

# Calculate IQR boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap outliers using IQR boundaries
df[numeric_columns] = df[numeric_columns].clip( lower=lower_bound, upper=upper_bound, axis="columns" )

# Null imputation 
###### (fit and trainsform to train and only transform to test (to prevent data leakage))

In [66]:
# Null imputation
# Median → numerical variables
# Mode   → categorical variables
# Calculate imputation values using training data only and imputating those values in both train and test data 

numeric_columns = train_x.select_dtypes(include=["int", "float"]).columns

categorical_columns = train_x.select_dtypes(include=["object", "category"]).columns

# Numerical columns → Median
for col in numeric_columns:
    median_value = train_x[col].median()

    train_x[col] = train_x[col].fillna(median_value)
    test_x[col] = test_x[col].fillna(median_value)

# Categorical columns → Mode
for col in categorical_columns:
    mode_value = train_x[col].mode()[0]

    train_x[col] = train_x[col].fillna(mode_value)
    test_x[col] = test_x[col].fillna(mode_value)

# Encoding 
###### (one hot encoding using pd.get_dummies())

In [67]:
# One-hot encode training data
train_x = pd.get_dummies( train_x, columns=categorical_columns, dtype=int )

# One-hot encode test data
test_x = pd.get_dummies( test_x, columns=categorical_columns, dtype=int )

# Ensure train and test have the same columns (-- this is important)
train_x, test_x = train_x.align( test_x, join="left", axis=1, fill_value=0 )

In [68]:
# Correcting column names for clarity in both train_x and test_x 
# coz column names must have changed after one hot encoding

# for train: 
train_x.columns = train_x.columns.str.lower().str.replace(" ", "_").str.replace("-", "_")

# for test:
test_x.columns = test_x.columns.str.lower().str.replace(" ", "_").str.replace("-", "_")

# Feature Selection / Feature Reduction

In [70]:
# Using SelectKBest

skb_feat = SelectKBest(score_func=f_classif, k=5)
skb_feat.fit(train_x, train_y)

skb_list = list((train_x.columns)[skb_feat.get_support()])
print(skb_list)

['distancekm', 'fuelconsumptionliters', 'historicontimerate', 'vehicleageyears', 'trafficcondition_high']


In [71]:
# Using RFE

rfe_feat = RFE(estimator=LogisticRegression(), n_features_to_select= 5)
rfe_feat.fit(train_x, train_y)


rfe_list = list((train_x.columns)[rfe_feat.get_support()])
print(rfe_list)

['historicontimerate', 'carrierid_10', 'shippingpriority_same_day', 'trafficcondition_high', 'weathercondition_storm']


In [72]:
# Selected variables
var_selected = list(set(rfe_list + skb_list))

# Standerdize the data

In [73]:
std = StandardScaler()
trainx = std.fit_transform(train_x)
testx = std.transform(test_x)

train_x = pd.DataFrame(trainx, columns=train_x.columns)
test_x = pd.DataFrame(testx, columns=test_x.columns)

In [74]:
train_x

,capacitykg,capacityunits,customertenureyears,distancekm,experienceyears,fragileflag,fuelconsumptionliters,historicontimerate,isholiday,isweekend,...,warehousename_warehouse_029,warehousename_warehouse_030,warehousetype_cross_dock,warehousetype_fulfillment_center,warehousetype_regional_hub,weathercondition_clear,weathercondition_extreme_heat,weathercondition_fog,weathercondition_rain,weathercondition_storm
0,0.525185,1.378994,0.119984,1.007688,-0.756736,-0.595038,0.254436,0.101926,-0.150449,-0.631266,...,-0.188613,-0.185413,-0.441807,-0.703111,0.988338,-1.121889,-0.296453,-0.333824,-0.492964,3.756008
1,-0.159285,1.332687,-0.329899,1.063766,-0.791754,1.680565,-0.065557,0.101926,-0.150449,-0.631266,...,-0.188613,-0.185413,-0.441807,-0.703111,0.988338,0.891354,-0.296453,-0.333824,-0.492964,-0.266240
2,-0.031423,-0.717654,-0.301664,0.893169,0.223753,1.680565,0.043766,-0.290736,-0.150449,1.584119,...,-0.188613,-0.185413,-0.441807,-0.703111,0.988338,0.891354,-0.296453,-0.333824,-0.492964,-0.266240
3,-0.737781,-1.436102,-0.952960,1.591094,-0.826771,-0.595038,2.197049,-0.552510,-0.150449,-0.631266,...,-0.188613,-0.185413,-0.441807,1.422250,-1.011800,0.891354,-0.296453,-0.333824,-0.492964,-0.266240
4,-0.371095,-0.201096,1.618341,-0.114699,-0.581649,-0.595038,-0.817207,0.494588,-0.150449,-0.631266,...,-0.188613,-0.185413,-0.441807,-0.703111,0.988338,-1.121889,-0.296453,-0.333824,-0.492964,3.756008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14317,0.049893,1.322947,-0.083310,1.226423,0.118700,-0.595038,1.323953,0.756362,-0.150449,1.584119,...,-0.188613,-0.185413,-0.441807,1.422250,-1.011800,0.891354,-0.296453,-0.333824,-0.492964,-0.266240
14318,1.714177,1.322947,1.290811,-0.074837,-1.071893,-0.595038,-0.577185,-0.552510,-0.150449,1.584119,...,-0.188613,-0.185413,-0.441807,1.422250,-1.011800,-1.121889,-0.296453,2.995586,-0.492964,-0.266240
14319,2.334924,-0.598212,1.494105,-1.067167,-0.056387,-0.595038,-0.893988,0.756362,-0.150449,1.584119,...,-0.188613,-0.185413,2.263433,-0.703111,-1.011800,-1.121889,-0.296453,-0.333824,2.028546,-0.266240
14320,0.071365,1.332687,-0.644253,-1.240297,-1.176946,-0.595038,-1.036916,1.149024,-0.150449,-0.631266,...,-0.188613,-0.185413,-0.441807,-0.703111,0.988338,-1.121889,-0.296453,2.995586,-0.492964,-0.266240


# Making data balanced

In [75]:
# Checking in-equality among y_variable in training data

train_y.value_counts()/train_y.count()

late_delivery
0    0.849043
1    0.150957
Name: count, dtype: float64

In [76]:
# Making data balanced

# Using Under sampling technique for this

ros = RandomOverSampler(random_state=1)
train_x, train_y = ros.fit_resample( train_x, train_y )

In [77]:
# Saving datasets as pickle so that they can be used further


train_x.to_pickle("C:/Users/Naveen/Desktop/GitHub/TransGlobe_Business/Classification_model/train_x.pkl")
train_y.to_pickle("C:/Users/Naveen/Desktop/GitHub/TransGlobe_Business/Classification_model/train_y.pkl")
test_x.to_pickle("C:/Users/Naveen/Desktop/GitHub/TransGlobe_Business/Classification_model/test_x.pkl")
test_y.to_pickle("C:/Users/Naveen/Desktop/GitHub/TransGlobe_Business/Classification_model/test_y.pkl")

# ----------------------------------------- END -----------------------------------------